In [1]:
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import List
from enum import Enum
import json
import xml.etree.ElementTree as ET
from cltk import NLP

In [ ]:
# Reuse of BDC / PSC chunking
class ChunkType(str, Enum):
    #SENTENCE = "sentence"
    #THREE_SENTENCE = "3-sentence"
    #TOKEN_WINDOW = "150-token-window"
    VERSE = "verse" 


@dataclass
class Chunk:
    chunk_id: str
    text: str
    chunk_type: ChunkType
    source_file: str
    position: int
    token_count: int
    sentence_indices: List[int] = field(default_factory=list)
    # Bible metadata
    book: str = ""
    chapter: int = 0
    verse: int = 0
    lemmatized: List[str] = field(default_factory=list)

    def to_dict(self):
        data = asdict(self)
        data["chunk_type"] = self.chunk_type.value
        return data


class VulgateChunker:
    """
    Chunks Vulgate XML into verse-level segments with CLTK lemmatization.
    """
    
    def __init__(self, nlp: NLP):
        self.nlp = nlp
        
    def parse_vulgate_xml(self, xml_path: Path) -> List[Chunk]:
        """
        Parse USFX XML and create verse-level chunks.
        """
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        chunks = []
        position = 0
        
        # Track current context
        current_book = ""
        current_book_id = ""
        current_chapter = 0
        
        for book in root.findall('book'):
            current_book_id = book.get('id', '')
            
            # Get book name from <h> tag
            h_tag = book.find('h')
            current_book = h_tag.text if h_tag is not None else current_book_id
            
            for elem in book:
                # Track chapter changes
                if elem.tag == 'c':
                    current_chapter = int(elem.get('id', 0))
                
                # Process verses
                elif elem.tag == 'v':
                    verse_id = elem.get('id', '0')
                    
                    # Extract verse text (text after <v> tag until <ve> or next tag)
                    verse_text = elem.tail if elem.tail else ""
                    verse_text = verse_text.strip()
                    
                    if not verse_text:
                        continue
                    
                    # Tokenize and lemmatize using CLTK
                    try:
                        doc = self.nlp.analyze(verse_text)
                        
                        # Extract tokens
                        tokens = []
                        lemmas = []
                        
                        for sentence in doc.sentences:
                            for word in sentence.words:
                                if word.string and word.string.strip():
                                    tokens.append(word.string)
                                    # Get lemma (fallback to original if not available)
                                    lemma = word.lemma if hasattr(word, 'lemma') and word.lemma else word.string.lower()
                                    lemmas.append(lemma)
                        
                        if not tokens:
                            continue
                        
                        # Create chunk
                        chunk_id = f"{current_book_id.lower()}_{current_chapter}_{verse_id}"
                        
                        chunks.append(
                            Chunk(
                                chunk_id=chunk_id,
                                text=verse_text,
                                chunk_type=ChunkType.VERSE,
                                source_file=xml_path.name,
                                position=position,
                                token_count=len(tokens),
                                book=current_book,
                                chapter=current_chapter,
                                verse=int(verse_id),
                                lemmatized=lemmas,
                            )
                        )
                        
                        position += 1
                        
                    except Exception as e:
                        logger.warning(f"⚠ Error processing {current_book} {current_chapter}:{verse_id} - {e}")
                        continue
        
        return chunks
    
    def chunk_vulgate(self, xml_path: Path, output_path: Path):
        """
        Process Vulgate XML and save chunks to JSON.
        """
        logger.info(f"Processing Vulgate: {xml_path.name}\n")
        
        chunks = self.parse_vulgate_xml(xml_path)
        
        # Statistics
        total_chunks = len(chunks)
        total_tokens = sum(c.token_count for c in chunks)
        avg_tokens = total_tokens / total_chunks if total_chunks else 0
        
        # Count books
        unique_books = len(set(c.book for c in chunks))
        
        output_data = {
            "metadata": {
                "corpus": "Vulgate",
                "chunking_strategy": "verse-level with CLTK lemmatization",
                "source_file": xml_path.name,
                "total_verses": total_chunks,
                "total_books": unique_books,
                "total_tokens": total_tokens,
                "avg_tokens_per_verse": avg_tokens,
            },
            "chunks": [c.to_dict() for c in chunks],
        }
        
        output_path.write_text(
            json.dumps(output_data, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        
        logger.info("VULGATE CHUNKING COMPLETE")
        logger.info(f"Total verses: {total_chunks}")
        logger.info(f"Average tokens per verse: {avg_tokens:.2f}")
        logger.info(f"Books processed: {unique_books}\n")
        
        return chunks

if __name__ == "__main__":
    from cltk import NLP
    import logging
    
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)
    
    # Initialize CLTK with Latin
    nlp = NLP(language="lat", suppress_banner=True)
    
    # Chunk Vulgate
    vulgate_chunker = VulgateChunker(nlp)
    
    vulgate_xml = Path("../data/lat-clementine.usfx.xml")
    output_json = Path("../data/vulgate_chunks.json")
    
    vulgate_chunker.chunk_vulgate(vulgate_xml, output_json)

INFO:__main__:Processing Vulgate: lat-clementine.usfx.xml

INFO:CLTK:Model for 'lat' / 'wiki' / 'vec' already present at '/Users/lenap/cltk_data/lat/embeddings/fasttext/wiki.la.vec' and ``overwrite=False``.
INFO:gensim.models.keyedvectors:loading projection weights from /Users/lenap/cltk_data/lat/embeddings/fasttext/wiki.la.vec
INFO:gensim.utils:KeyedVectors lifecycle event {'msg': 'loaded (138974, 300) matrix of type float32 from /Users/lenap/cltk_data/lat/embeddings/fasttext/wiki.la.vec', 'binary': False, 'encoding': 'utf8', 'datetime': '2026-02-14T13:33:50.012875', 'gensim': '4.3.3', 'python': '3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]', 'platform': 'macOS-26.2-arm64-arm-64bit', 'event': 'load_word2vec_format'}


Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error

INFO:__main__:VULGATE CHUNKING COMPLETE
INFO:__main__:Total verses: 35809
INFO:__main__:Average tokens per verse: 20.88
INFO:__main__:Books processed: 73

